In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from phd_helpers.AbaqusPostprocessing import (
    inp2pv, get_field_path, get_field_df, add_field_to_mesh, get_history_path, parse_final_step_time
)

In [2]:
def get_run_info(study, param_path):
     return {
            'study': study,
            'run_id': param_path.with_suffix('').name
        }

def get_nested(d, key, default=None):
    for part in key.split("."):
        d = d.get(part, default) if isinstance(d, dict) else default
    return d

def get_run_params(param_path, loop_params):

        with open(param_path, 'r') as f:
            params = json.load(f)

        return {k: get_nested(params, k, np.nan) for k in loop_params}
    
def get_run_results(path, run_id, sub, bone, run_id_mesh, pose, step, frame, field_metrics):

    job = f'{run_id_mesh}-{pose}-{run_id}'
    inp_file = path / f'inpFiles/{sub}/inp/{job}/{job}.inp'
    csv_dir = inp_file.parent / 'resultCSVs' 

    # get meshes
    meshes = inp2pv(inp_file)
    mesh = meshes[bone]
    instance = f"{bone.upper()}_INST"

    # Field data
    for metric in field_metrics:
        field_path = get_field_path(csv_dir, metric, step, frame, instance)
        field_df = get_field_df(field_path)
        add_field_to_mesh(mesh, field_df)

    # History data
    history_data = pd.read_csv(get_history_path(csv_dir, step))
    RF_data = history_data[history_data['historyOutputKey']=='RF1']
    CAREA_data = history_data[history_data['historyOutputDescription']=='Total area in contact']

    # step data
    sta_path = inp_file.with_suffix('.sta')

    return {
        'RF': np.abs(RF_data['value'].iloc[frame]),
        'CA': CAREA_data['value'].iloc[frame],
        'P_max': mesh['CPRESS'].max(),
        'P_avg': np.mean(mesh['CPRESS'][mesh['CPRESS']>0]),
        #'loc_Pmax': mesh.points[np.argmax(mesh['CPRESS'])],
        'd' : parse_final_step_time(sta_path)
    }

def get_run_results_mesh(path, run_id, sub, bone, run_id_mesh, pose, step, frame, field_metrics):

    job = f'{run_id_mesh}-{pose}-{run_id}'
    inp_file = path / f'inpFiles/{sub}/inp/{job}/{job}.inp'
    csv_dir = inp_file.parent / 'resultCSVs' 

    # get meshes
    meshes = inp2pv(inp_file)
    mesh = meshes[bone]
    instance = f"{bone.upper()}_INST"

    # Field data
    for metric in field_metrics:
        field_path = get_field_path(csv_dir, metric, step, frame, instance)
        field_df = get_field_df(field_path)
        add_field_to_mesh(mesh, field_df)

    return mesh


In [219]:
# input data
inp_dict = {
    'sub': '14548R',
    'run_id_mesh': '35T',
    'pose': 'neutral',
    'bone': 'tpm',
    'step': 0,
    'frame': -1,
    'field_metrics': ["CPRESS", 'U'],
    #'history_metrics': ['CAREA', 'RF']
}

# list of investigated parameters
loop_params = [
    'bone_material.E', 'bone_material.nu', 
    'cartilage_friction', 'cartilage_material.D1',  'cartilage_material.C10', #'cartilage_material.model',
    ]

studies = [1, 2, 3, 4]

data = []
for study in studies:
    path = Path(f'../../../Computational/InpPipeline/outputs/initialFEAstuff/sensitivity/study{study}_35T4d5')
    param_paths = path.glob('params/loop_params/*.json')

    for param_path in param_paths:
        info_dict = get_run_info(study, param_path)
        param_dict = get_run_params(param_path, loop_params)
        result_dict = get_run_results(path=path, run_id=info_dict['run_id'], **inp_dict)

        data.append(info_dict | param_dict | result_dict)


df = pd.DataFrame(data).sort_values(['study', 'run_id']).reset_index(drop=True)
df['tensile'] = np.full(len(df), np.nan)
df.loc[(df['study']==4), 'tensile'] = 0.25
df.loc[(df['study']==2) | (df['study']==3), 'tensile'] = 4.0

loop_params += ['tensile']

In [226]:
mask = df['RF'] > 49

In [227]:
df

,study,run_id,bone_material.E,bone_material.nu,cartilage_friction,cartilage_material.D1,cartilage_material.C10,RF,CA,P_max,P_avg,d,tensile
0,1,00,1500,0.25,0.0,0.0,0.085,50.007751,52.412712,2.589318,0.793261,0.391,NaN
1,1,01,1500,0.25,1.0,0.0,0.085,49.998531,52.023518,2.589264,0.791678,0.388,NaN
2,1,02,1500,0.25,0.0,1.0,0.085,50.007938,63.185146,2.940717,0.750057,0.702,NaN
3,1,03,1500,0.25,1.0,1.0,0.085,26.284254,52.088802,1.428751,0.460954,0.522,NaN
4,1,04,1500,0.25,0.0,0.0,0.110,49.998020,48.540871,2.775532,0.865280,0.360,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,4,11,3000,0.25,1.0,1.0,NaN,28.542904,52.474499,1.579906,0.497966,0.530,0.25
60,4,12,3000,0.45,0.0,0.0,NaN,50.000957,54.118725,2.531687,0.773784,0.404,0.25
61,4,13,3000,0.45,1.0,0.0,NaN,46.018944,52.249348,2.414954,0.736083,0.386,0.25
62,4,14,3000,0.45,0.0,1.0,NaN,50.020802,62.382038,2.907595,0.762917,0.685,0.25


In [228]:
def compute_avg_change(df_clean, metric):
    return df_clean.groupby(param)[metric].mean().diff().iloc[1]


metric = 'P_max'

df50 = df[df['RF'] > 49] # remove any runs that failed before 50 N
changes = []
for param in loop_params:
    df_clean = df50[~df50[param].isna()] # remove any runs that didn't use that parameter

    changes.append({
        'param': param,
        'change': compute_avg_change(df_clean, metric)
    })
pd.DataFrame(changes)

,param,change
0,bone_material.E,0.019811
1,bone_material.nu,0.003768
2,cartilage_friction,-0.110434
3,cartilage_material.D1,0.309984
4,cartilage_material.C10,0.131866
5,tensile,0.271898


In [23]:
def paired_parameter_sensitivity(data, parameters, metric):
    """
    df : pandas.DataFrame
        Sensitivity study results.
    parameters : list[str]
        Input parameters to investigate.
    metric : str
        Output metric to compare, e.g. "P_max", "CA", "RF".
    """

    results = []

    df = data.copy()
    for param in parameters:
        levels = sorted(df[param].dropna().unique())

        if len(levels) != 2:
            raise ValueError(
                f"{param} has {len(levels)} levels, expected exactly 2: {levels}"
            )

        low, high = levels

        matching_cols = [p for p in parameters if p != param]
        if param == 'cartilage_material.model':
            matching_cols = [x for x in matching_cols if x!='cartilage_material.C10']
            df = data[data['cartilage_material.C10']!=0.085]

        low_df = df[df[param] == low].copy()
        high_df = df[df[param] == high].copy()

        merged = pd.merge(
            low_df,
            high_df,
            on=matching_cols,
            suffixes=("_low", "_high"),
        )

        changes = merged[f"{metric}_high"] - merged[f"{metric}_low"]


        results.append({
            "parameter": param,
            "low_value": low,
            "high_value": high,
            "n_pairs": len(changes),
            "max_change": changes.max(),
            "mean_change": changes.mean(),
            "min_change": changes.min()
        })

        if param == 'cartilage_material.model':
            df = data.copy()

    return pd.DataFrame(results)

In [230]:
df50 = df[mask] # remove any runs that failed before 50 N
summary = paired_parameter_sensitivity(df50, loop_params, "P_max")
summary
# could report % change relative to accuracy box results

,parameter,low_value,high_value,n_pairs,max_change,mean_change,min_change
0,bone_material.E,1500.000,3000.00,22,0.044729,0.019811,0.007338
1,bone_material.nu,0.250,0.45,22,0.013657,0.003768,0.000713
2,cartilage_friction,0.000,1.00,12,0.010059,0.003356,-0.001324
3,cartilage_material.D1,0.000,1.00,16,0.426082,0.334288,0.182081
4,cartilage_material.C10,0.085,0.11,12,0.196567,0.131866,0.009487
5,tensile,0.250,4.00,8,0.372013,0.340024,0.314790


In [231]:
df50 = df[mask] # remove any runs that failed before 50 N
summary = paired_parameter_sensitivity(df50, loop_params, "CA")
summary

,parameter,low_value,high_value,n_pairs,max_change,mean_change,min_change
0,bone_material.E,1500.000,3000.00,22,-0.017574,-0.115555,-0.235844
1,bone_material.nu,0.250,0.45,22,0.013767,-0.008403,-0.039234
2,cartilage_friction,0.000,1.00,12,-0.303661,-0.384555,-0.438126
3,cartilage_material.D1,0.000,1.00,16,13.928642,11.275549,8.263313
4,cartilage_material.C10,0.085,0.11,12,-2.461472,-3.404425,-3.881104
5,tensile,0.250,4.00,8,-1.294144,-4.117545,-6.959473


In [22]:
def paired_sensitivity_details(
    df,
    parameters,
    metric,
    target_parameter,
    change_type="absolute",
):
    """
    Returns row-by-row paired changes for one parameter in a 2-level full-factorial study.
    """

    levels = sorted(df[target_parameter].dropna().unique())

    if len(levels) != 2:
        raise ValueError(
            f"{target_parameter} has {len(levels)} levels, expected exactly 2."
        )

    low, high = levels

    matching_cols = [p for p in parameters if p != target_parameter]

    low_df = df[df[target_parameter] == low].copy()
    high_df = df[df[target_parameter] == high].copy()

    paired = pd.merge(
        low_df,
        high_df,
        on=matching_cols,
        suffixes=("_low", "_high"),
    )

    if change_type == "absolute":
        paired["change"] = paired[f"{metric}_high"] - paired[f"{metric}_low"]

    elif change_type == "percent":
        paired["change"] = 100 * (
            paired[f"{metric}_high"] - paired[f"{metric}_low"]
        ) / paired[f"{metric}_low"]

    elif change_type == "relative":
        paired["change"] = (
            paired[f"{metric}_high"] - paired[f"{metric}_low"]
        ) / paired[f"{metric}_low"]

    else:
        raise ValueError("change_type must be 'absolute', 'percent', or 'relative'")

    paired["parameter"] = target_parameter
    paired["low_value"] = low
    paired["high_value"] = high

    output_cols = (
        ["parameter", "low_value", "high_value"]
        + matching_cols
        + [
            f"{metric}_low",
            f"{metric}_high",
            "change",
        ]
    )

    return paired[output_cols].sort_values("change", ascending=False)

In [232]:
metric = "P_max"

D1_pairs = paired_sensitivity_details(
    df=df[mask],
    parameters=loop_params,
    metric=metric,
    target_parameter="cartilage_material.D1",
    change_type="absolute",
)

C10_pairs = paired_sensitivity_details(
    df=df[mask],
    parameters=loop_params,
    metric=metric,
    target_parameter="cartilage_material.C10",
    change_type="absolute",
)

In [233]:
C10_pairs

,parameter,low_value,high_value,bone_material.E,bone_material.nu,cartilage_friction,cartilage_material.D1,tensile,P_max_low,P_max_high,change
10,cartilage_material.C10,0.085,0.11,3000,0.45,1.0,0.0,NaN,2.601839,2.798406,0.196567
7,cartilage_material.C10,0.085,0.11,3000,0.25,1.0,0.0,NaN,2.601068,2.797516,0.196448
1,cartilage_material.C10,0.085,0.11,1500,0.25,1.0,0.0,NaN,2.589264,2.783584,0.194320
4,cartilage_material.C10,0.085,0.11,1500,0.45,1.0,0.0,NaN,2.591522,2.785323,0.193800
9,cartilage_material.C10,0.085,0.11,3000,0.45,0.0,0.0,NaN,2.600172,2.788348,0.188176
6,cartilage_material.C10,0.085,0.11,3000,0.25,0.0,0.0,NaN,2.599375,2.787457,0.188082
3,cartilage_material.C10,0.085,0.11,1500,0.45,0.0,0.0,NaN,2.590512,2.777225,0.186713
0,cartilage_material.C10,0.085,0.11,1500,0.25,0.0,0.0,NaN,2.589318,2.775532,0.186214
2,cartilage_material.C10,0.085,0.11,1500,0.25,0.0,1.0,NaN,2.940717,2.957613,0.016896
5,cartilage_material.C10,0.085,0.11,1500,0.45,0.0,1.0,NaN,2.950583,2.966559,0.015975


In [234]:
D1_pairs

,parameter,low_value,high_value,bone_material.E,bone_material.nu,cartilage_friction,cartilage_material.C10,tensile,P_max_low,P_max_high,change
11,cartilage_material.D1,0.0,1.0,3000,0.45,0.0,NaN,4.00,2.853527,3.279609,0.426082
10,cartilage_material.D1,0.0,1.0,3000,0.25,0.0,NaN,4.00,2.852258,3.271185,0.418927
9,cartilage_material.D1,0.0,1.0,1500,0.45,0.0,NaN,4.00,2.839138,3.240112,0.400974
8,cartilage_material.D1,0.0,1.0,1500,0.25,0.0,NaN,4.00,2.838040,3.226456,0.388416
6,cartilage_material.D1,0.0,1.0,3000,0.45,0.0,0.085,NaN,2.600172,2.984307,0.384135
4,cartilage_material.D1,0.0,1.0,3000,0.25,0.0,0.085,NaN,2.599375,2.979329,0.379954
15,cartilage_material.D1,0.0,1.0,3000,0.45,0.0,NaN,0.25,2.531687,2.907595,0.375909
14,cartilage_material.D1,0.0,1.0,3000,0.25,0.0,NaN,0.25,2.530974,2.903902,0.372927
13,cartilage_material.D1,0.0,1.0,1500,0.45,0.0,NaN,0.25,2.524348,2.884826,0.360477
2,cartilage_material.D1,0.0,1.0,1500,0.45,0.0,0.085,NaN,2.590512,2.950583,0.360071


# Study 5. - ! WRONG MATERIAL MODELS N = 1 !

In [247]:
# input data
inp_dict = {
    'sub': '14548R',
    'run_id_mesh': '35T',
    'pose': 'neutral',
    'bone': 'tpm',
    'step': 0,
    'frame': -1,
    'field_metrics': ["CPRESS", 'U'],
    #'history_metrics': ['CAREA', 'RF']
}

# list of investigated parameters
loop_params5 = [
    'bone_material.E', 'bone_material.nu', 
    'cartilage_friction', 'cartilage_material.D1',  
    ]

studies = ['5a', '5b', '5c', '5d']

data = []
for study in studies:
    path = Path(f'../../../Computational/InpPipeline/outputs/initialFEAstuff/sensitivity/study{study}_35T4d5')
    param_paths = path.glob('params/loop_params/*.json')

    for param_path in param_paths:
        try:
            info_dict = get_run_info(study, param_path)
            param_dict = get_run_params(param_path, loop_params5)
            result_dict = get_run_results(path=path, run_id=info_dict['run_id'], **inp_dict)

            data.append(info_dict | param_dict | result_dict)
        except:
            continue


df5 = pd.DataFrame(data).sort_values(['study', 'run_id']).reset_index(drop=True)
df5['tensile'] = np.full(len(df5), np.nan)
df5['compressive'] = np.full(len(df5), np.nan)
df5.loc[(df5['study']=='5a'), 'compressive'] = 1.0
df5.loc[(df5['study']=='5b'), 'compressive'] = 2.0
df5.loc[(df5['study']=='5c'), 'tensile'] = 1.0
df5.loc[(df5['study']=='5d'), 'tensile'] = 2.0

df5.loc[df5['cartilage_material.D1']>0, 'cartilage_material.D1'] = 1.0

loop_params5 += ['compressive', 'tensile']

In [ ]:
mask = df5['RF'] > 49

In [250]:
df5m = df5[mask]
df5m

,study,run_id,bone_material.E,bone_material.nu,cartilage_friction,cartilage_material.D1,RF,CA,P_max,P_avg,d,tensile,compressive
0,5a,00,1400,0.25,0.0,0.0,50.044952,52.647846,2.576672,0.786594,0.394,NaN,1.0
2,5a,02,1400,0.25,0.0,1.0,50.006706,63.691612,3.027871,0.743220,0.712,NaN,1.0
4,5a,04,1400,0.45,0.0,0.0,50.029015,52.652554,2.577723,0.785869,0.393,NaN,1.0
5,5a,10,3000,0.25,0.0,1.0,50.033684,63.651882,3.077130,0.743831,0.707,NaN,1.0
7,5a,12,3000,0.45,0.0,0.0,49.997364,52.548187,2.587936,0.786845,0.388,NaN,1.0
8,5a,13,3000,0.45,1.0,0.0,49.999119,52.112087,2.580256,0.785166,0.384,NaN,1.0
9,5a,14,3000,0.45,0.0,1.0,50.029285,63.631508,3.082938,0.743785,0.706,NaN,1.0
11,5b,00,1400,0.25,0.0,0.0,50.036621,47.650646,2.826453,0.886054,0.353,NaN,2.0
12,5b,01,1400,0.25,1.0,0.0,49.999405,47.268826,2.830449,0.886932,0.350,NaN,2.0
13,5b,02,1400,0.25,0.0,1.0,50.047104,60.368443,3.092701,0.795112,0.689,NaN,2.0


In [251]:
summary = paired_parameter_sensitivity(df5m, loop_params5, "P_max")
summary

,parameter,low_value,high_value,n_pairs,max_change,mean_change,min_change
0,bone_material.E,1400.00,3000.00,15,0.116578,0.058874,0.010213
1,bone_material.nu,0.25,0.45,15,0.024840,0.008105,0.000911
2,cartilage_friction,0.00,1.00,4,0.012478,0.004924,-0.007680
3,cartilage_material.D1,0.00,1.00,14,0.495002,-1.269434,-2.760231
4,compressive,1.00,2.00,6,0.251836,0.154827,0.056084
5,tensile,1.00,2.00,8,0.135402,-0.119444,-0.375754


In [255]:
summary = paired_parameter_sensitivity(df5m, loop_params5, "CA")
summary

,parameter,low_value,high_value,n_pairs,max_change,mean_change,min_change
0,bone_material.E,1400.00,3000.00,15,-0.039730,-0.244460,-0.402061
1,bone_material.nu,0.25,0.45,15,0.013378,-0.020097,-0.133461
2,cartilage_friction,0.00,1.00,4,-0.356205,-0.386342,-0.436100
3,cartilage_material.D1,0.00,1.00,14,18.587210,15.544245,11.043766
4,compressive,1.00,2.00,6,-3.323170,-4.234584,-5.166054
5,tensile,1.00,2.00,8,2.221729,1.731488,1.290945


In [261]:
metric = "P_max"

D1_pairs = paired_sensitivity_details(
    df=df5m,
    parameters=loop_params5,
    metric=metric,
    target_parameter="cartilage_material.D1",
    change_type="absolute",
)

comp_pairs = paired_sensitivity_details(
    df=df5m,
    parameters=loop_params5,
    metric=metric,
    target_parameter="compressive",
    change_type="absolute",
)

tens_pairs = paired_sensitivity_details(
    df=df5m,
    parameters=loop_params5,
    metric=metric,
    target_parameter="tensile",
    change_type="absolute",
)

In [262]:
tens_pairs

,parameter,low_value,high_value,bone_material.E,bone_material.nu,cartilage_friction,cartilage_material.D1,compressive,P_max_low,P_max_high,change
7,tensile,1.0,2.0,3000,0.45,0.0,1.0,NaN,4.506921,4.642323,0.135402
5,tensile,1.0,2.0,3000,0.25,0.0,1.0,NaN,4.497141,4.631838,0.134697
3,tensile,1.0,2.0,1400,0.45,0.0,1.0,NaN,4.450341,4.576044,0.125703
1,tensile,1.0,2.0,1400,0.25,0.0,1.0,NaN,4.429732,4.551204,0.121472
2,tensile,1.0,2.0,1400,0.45,0.0,0.0,NaN,7.147442,6.786966,-0.360476
0,tensile,1.0,2.0,1400,0.25,0.0,0.0,NaN,7.140794,6.779102,-0.361692
6,tensile,1.0,2.0,3000,0.45,0.0,0.0,NaN,7.260737,6.885832,-0.374905
4,tensile,1.0,2.0,3000,0.25,0.0,0.0,NaN,7.257372,6.881618,-0.375754


In [257]:
# compressive behaviour matters if incompressible
comp_pairs

,parameter,low_value,high_value,bone_material.E,bone_material.nu,cartilage_friction,cartilage_material.D1,tensile,P_max_low,P_max_high,change
4,compressive,1.0,2.0,3000,0.45,0.0,0.0,NaN,2.587936,2.839772,0.251836
2,compressive,1.0,2.0,1400,0.45,0.0,0.0,NaN,2.577723,2.827767,0.250043
0,compressive,1.0,2.0,1400,0.25,0.0,0.0,NaN,2.576672,2.826453,0.249781
1,compressive,1.0,2.0,1400,0.25,0.0,1.0,NaN,3.027871,3.092701,0.064830
3,compressive,1.0,2.0,3000,0.25,0.0,1.0,NaN,3.077130,3.133516,0.056386
5,compressive,1.0,2.0,3000,0.45,0.0,1.0,NaN,3.082938,3.139023,0.056084


In [ ]:
# D1 always matters but twice as much if cartilage is stiffer and even more when shear? behaviour is different
D1_pairs

,parameter,low_value,high_value,bone_material.E,bone_material.nu,cartilage_friction,compressive,tensile,P_max_low,P_max_high,change
1,cartilage_material.D1,0.0,1.0,3000,0.45,0.0,1.0,NaN,2.587936,3.082938,0.495002
0,cartilage_material.D1,0.0,1.0,1400,0.25,0.0,1.0,NaN,2.576672,3.027871,0.451199
5,cartilage_material.D1,0.0,1.0,3000,0.45,0.0,2.0,NaN,2.839772,3.139023,0.299250
4,cartilage_material.D1,0.0,1.0,3000,0.25,0.0,2.0,NaN,2.838862,3.133516,0.294654
3,cartilage_material.D1,0.0,1.0,1400,0.45,0.0,2.0,NaN,2.827767,3.103657,0.275890
2,cartilage_material.D1,0.0,1.0,1400,0.25,0.0,2.0,NaN,2.826453,3.092701,0.266248
11,cartilage_material.D1,0.0,1.0,1400,0.45,0.0,NaN,2.0,6.786966,4.576044,-2.210922
10,cartilage_material.D1,0.0,1.0,1400,0.25,0.0,NaN,2.0,6.779102,4.551204,-2.227898
13,cartilage_material.D1,0.0,1.0,3000,0.45,0.0,NaN,2.0,6.885832,4.642323,-2.243509
12,cartilage_material.D1,0.0,1.0,3000,0.25,0.0,NaN,2.0,6.881618,4.631838,-2.249780


# Study 6
- Redo 5 with n = 2

In [6]:
studies = ['6a', '6b', '6c', '6e']

data6 = []
for study in studies:
    path = Path(f'../../../Computational/InpPipeline/outputs/initialFEAstuff/sensitivity/study{study}_35T4d5')
    param_path = path / 'params/full_params.json'
    with open(param_path, 'r') as f:
        params = json.load(f)

    data6.append({study: params['inp']['cartilage_material']})

In [7]:
data6

[{'6a': {'model': 'ogden',
   'n': 2,
   'mu1': 0.1617,
   'alpha1': 2.1968,
   'mu2': 0.0404,
   'alpha2': -0.8768,
   'D1': [0.0, 1.0238],
   'D2': 0.0}},
 {'6b': {'model': 'ogden',
   'n': 2,
   'mu1': 0.2398,
   'alpha1': 1.7921,
   'mu2': -0.052,
   'alpha2': -0.6156,
   'D1': [0.0, 1.1016],
   'D2': 0.0}},
 {'6c': {'model': 'ogden',
   'n': 2,
   'mu1': 3.9,
   'alpha1': 1.4569,
   'mu2': -3.6845,
   'alpha2': 1.3396,
   'D1': [0.0, 0.9603],
   'D2': 0.0}},
 {'6e': {'model': 'ogden',
   'n': 2,
   'mu1': 0.1774,
   'alpha1': 0.674,
   'mu2': 0.0026,
   'alpha2': 0.6738,
   'D1': [0.0, 1.1495],
   'D2': 0.0}}]

In [67]:
studyN = 6

# input data
inp_dict = {
    'sub': '14548R',
    'run_id_mesh': '35T',
    'pose': 'neutral',
    'bone': 'tpm',
    'step': 0,
    'frame': -1,
    'field_metrics': ["CPRESS", 'U'],
    #'history_metrics': ['CAREA', 'RF']
}

# list of investigated parameters
loop_params6 = [
    'bone_material.E', 'bone_material.nu', 
    'cartilage_material.D1',  'cartilage_friction', 
    ]

studies = [f'{studyN}a', f'{studyN}b', f'{studyN}c', f'{studyN}e']

data6 = []
for study in studies:
    path = Path(f'../../../Computational/InpPipeline/outputs/initialFEAstuff/sensitivity/study{study}_35T4d5')
    param_paths = path.glob('params/loop_params/*.json')

    for param_path in param_paths:
        try:
            info_dict = get_run_info(study, param_path)
            param_dict = get_run_params(param_path, loop_params6)
            result_dict = get_run_results(path=path, run_id=info_dict['run_id'], **inp_dict)

            data6.append(info_dict | param_dict | result_dict)
        except:
            continue

df6 = pd.DataFrame(data6).sort_values(['study', 'run_id']).reset_index(drop=True)
df6['tensile'] = np.full(len(df6), np.nan)
df6['compressive'] = np.full(len(df6), np.nan)
df6.loc[(df6['study']==f'{studyN}a'), 'compressive'] = 1.0 # soft
df6.loc[(df6['study']==f'{studyN}b'), 'compressive'] = 2.0 # stiff
df6.loc[(df6['study']==f'{studyN}c'), 'tensile'] = 2.0 # stiff
df6.loc[(df6['study']==f'{studyN}e'), 'tensile'] = 1.0 # soft

df6.loc[df6['cartilage_material.D1']>0, 'cartilage_material.D1'] = 1.0

loop_params6 += ['compressive', 'tensile']

In [ ]:
mask6 = df6['RF'] > 49
df6m = df6[mask6]
df6m

,study,run_id,bone_material.E,bone_material.nu,cartilage_material.D1,cartilage_friction,RF,CA,P_max,P_avg,d,tensile,compressive
0,6a,00,1400,0.25,0.0,0.0,49.997375,49.428928,2.746972,0.848726,0.368,NaN,1.0
1,6a,01,1400,0.25,0.0,1.0,49.999519,49.312248,2.755472,0.843846,0.366,NaN,1.0
2,6a,02,1400,0.25,1.0,0.0,50.032009,61.312641,3.089462,0.779189,0.687,NaN,1.0
4,6a,04,1400,0.45,0.0,0.0,49.998119,49.436646,2.748958,0.848202,0.367,NaN,1.0
5,6a,05,1400,0.45,0.0,1.0,49.962624,49.204521,2.755957,0.845046,0.364,NaN,1.0
6,6a,06,1400,0.45,1.0,0.0,50.023228,61.290668,3.100895,0.778600,0.685,NaN,1.0
8,6a,08,3000,0.25,0.0,0.0,50.035831,49.357372,2.762396,0.848056,0.363,NaN,1.0
9,6a,09,3000,0.25,0.0,1.0,50.018776,49.014599,2.772277,0.847637,0.360,NaN,1.0
10,6a,10,3000,0.25,1.0,0.0,50.005322,61.194508,3.131665,0.780937,0.682,NaN,1.0
12,6a,12,3000,0.45,0.0,0.0,50.027802,49.346157,2.762998,0.848816,0.362,NaN,1.0


In [70]:
summary = paired_parameter_sensitivity(df6m, loop_params6, "P_max")
summary

,parameter,low_value,high_value,n_pairs,max_change,mean_change,min_change
0,bone_material.E,1400.00,3000.00,17,0.043537,0.025181,0.009609
1,bone_material.nu,0.25,0.45,17,0.015126,0.004981,0.000484
2,cartilage_material.D1,0.00,1.00,16,0.560357,0.379981,0.167084
3,cartilage_friction,0.00,1.00,3,0.009881,0.008460,0.006998
4,compressive,1.00,2.00,8,-0.062184,-0.097965,-0.135459
5,tensile,1.00,2.00,8,0.255980,0.074636,-0.104812


In [71]:
summary = paired_parameter_sensitivity(df6m, loop_params6, "CA")
summary

,parameter,low_value,high_value,n_pairs,max_change,mean_change,min_change
0,bone_material.E,1400.00,3000.00,17,-0.066589,-0.113341,-0.297649
1,bone_material.nu,0.25,0.45,17,0.015366,-0.007813,-0.107727
2,cartilage_material.D1,0.00,1.00,16,13.250454,11.474901,9.145699
3,cartilage_friction,0.00,1.00,3,-0.116680,-0.230526,-0.342773
4,compressive,1.00,2.00,8,1.966690,1.824173,1.686180
5,tensile,1.00,2.00,8,-2.032028,-4.072171,-6.133678


In [72]:
metric = "P_max"

D1_pairs = paired_sensitivity_details(
    df=df6m,
    parameters=loop_params6,
    metric=metric,
    target_parameter="cartilage_material.D1",
    change_type="absolute",
)

comp_pairs = paired_sensitivity_details(
    df=df6m,
    parameters=loop_params6,
    metric=metric,
    target_parameter="compressive",
    change_type="absolute",
)

#tens_pairs = paired_sensitivity_details(
#    df=df6m,
#    parameters=loop_params6,
#    target_parameter="tensile",
#    change_type="absolute",
#    metric=metric,
#)

In [73]:
comp_pairs

,parameter,low_value,high_value,bone_material.E,bone_material.nu,cartilage_material.D1,cartilage_friction,tensile,P_max_low,P_max_high,change
7,compressive,1.0,2.0,3000,0.45,1.0,0.0,NaN,3.137422,3.075238,-0.062184
5,compressive,1.0,2.0,3000,0.25,1.0,0.0,NaN,3.131665,3.069372,-0.062293
3,compressive,1.0,2.0,1400,0.45,1.0,0.0,NaN,3.100895,3.038116,-0.062779
1,compressive,1.0,2.0,1400,0.25,1.0,0.0,NaN,3.089462,3.026416,-0.063046
0,compressive,1.0,2.0,1400,0.25,0.0,0.0,NaN,2.746972,2.616079,-0.130893
2,compressive,1.0,2.0,1400,0.45,0.0,0.0,NaN,2.748958,2.617203,-0.131755
6,compressive,1.0,2.0,3000,0.45,0.0,0.0,NaN,2.762998,2.627689,-0.135309
4,compressive,1.0,2.0,3000,0.25,0.0,0.0,NaN,2.762396,2.626937,-0.135459


# Study 7

In [88]:
studyN = 7
studies = [f'{studyN}a', f'{studyN}b', f'{studyN}c', f'{studyN}d']

data6 = []
for study in studies:
    path = Path(f'../../../Computational/InpPipeline/outputs/initialFEAstuff/sensitivity/study{study}_35T4d5')
    param_path = path / 'params/full_params.json'
    with open(param_path, 'r') as f:
        params = json.load(f)

    data6.append({study: params['inp']['cartilage_material']})

In [89]:
data6

[{'7a': {'model': 'ogden',
   'n': 2,
   'mu1': 0.18101,
   'alpha1': 2.04319,
   'mu2': 0.02742,
   'alpha2': -0.83163,
   'D1': [0.0, 0.99265],
   'D2': 0.0}},
 {'7b': {'model': 'ogden',
   'n': 2,
   'mu1': 0.2084,
   'alpha1': 1.96173,
   'mu2': -0.02591,
   'alpha2': -0.82933,
   'D1': [0.0, 1.13376],
   'D2': 0.0}},
 {'7c': {'model': 'ogden',
   'n': 2,
   'mu1': 0.21504,
   'alpha1': 2.08646,
   'mu2': -0.01141,
   'alpha2': -1.26391,
   'D1': [0.0, 1.01604],
   'D2': 0.0}},
 {'7d': {'model': 'ogden',
   'n': 2,
   'mu1': 0.1729,
   'alpha1': 1.88654,
   'mu2': 0.01265,
   'alpha2': -1.22573,
   'D1': [0.0, 1.11507],
   'D2': 0.0}}]

In [ ]:
# input data
inp_dict = {
    'sub': '14548R',
    'run_id_mesh': '35T',
    'pose': 'neutral',
    'bone': 'tpm',
    'step': 0,
    'frame': -1,
    'field_metrics': ["CPRESS", 'U'],
    #'history_metrics': ['CAREA', 'RF']
}

# list of investigated parameters
loop_params6 = [
    'bone_material.E', 'bone_material.nu', 
    'cartilage_material.D1',  'cartilage_friction', 
    ]

studies = [f'{studyN}a', f'{studyN}b', f'{studyN}c', f'{studyN}e']

data6 = []
for study in studies:
    path = Path(f'../../../Computational/InpPipeline/outputs/initialFEAstuff/sensitivity/study{study}_35T4d5')
    param_paths = path.glob('params/loop_params/*.json')

    for param_path in param_paths:
        try:
            info_dict = get_run_info(study, param_path)
            param_dict = get_run_params(param_path, loop_params6)
            result_dict = get_run_results(path=path, run_id=info_dict['run_id'], **inp_dict)

            data6.append(info_dict | param_dict | result_dict)
        except:
            continue

df6 = pd.DataFrame(data6).sort_values(['study', 'run_id']).reset_index(drop=True)
df6['tensile'] = np.full(len(df6), np.nan)
df6['compressive'] = np.full(len(df6), np.nan)
df6.loc[(df6['study']==f'{studyN}a'), 'compressive'] = 1.0 # soft
df6.loc[(df6['study']==f'{studyN}b'), 'compressive'] = 2.0 # stiff
df6.loc[(df6['study']==f'{studyN}c'), 'tensile'] = 2.0 # stiff
df6.loc[(df6['study']==f'{studyN}d'), 'tensile'] = 1.0 # soft

df6.loc[df6['cartilage_material.D1']>0, 'cartilage_material.D1'] = 1.0

loop_params6 += ['compressive', 'tensile']

In [ ]:
mask6 = df6['RF'] > 49
df6m = df6[mask6]
df6m

In [ ]:
summary = paired_parameter_sensitivity(df6m, loop_params6, "P_max")
summary

# Strain magnitude

In [65]:
def get_field_df_wrap(sub, study, run_id, run_id_mesh, pose, bone, step, frame, field):
    path = Path(f'../../../Computational/InpPipeline/outputs/initialFEAstuff/sensitivity/study{study}_35T4d5')
    job = f'{run_id_mesh}-{pose}-{run_id}'
    inp_file = path / f'inpFiles/{sub}/inp/{job}/{job}.inp'
    csv_dir = inp_file.parent / 'resultCSVs' 
    instance = f"{bone.upper()}_INST"
    
    field_path = get_field_path(csv_dir, field, step, frame, instance)
    return get_field_df(field_path)

def add_strain_measures_from_LE(df):
    n = len(df)

    LE = np.zeros((n, 3, 3), dtype=float)

    # Abaqus tensor order:
    # LE1 = LE11, LE2 = LE22, LE3 = LE33
    # LE4 = LE12, LE5 = LE13, LE6 = LE23
    LE[:, 0, 0] = df["LE1"]
    LE[:, 1, 1] = df["LE2"]
    LE[:, 2, 2] = df["LE3"]

    LE[:, 0, 1] = LE[:, 1, 0] = df["LE4"]
    LE[:, 0, 2] = LE[:, 2, 0] = df["LE5"]
    LE[:, 1, 2] = LE[:, 2, 1] = df["LE6"]

    # Principal logarithmic strains, sorted ascending
    principal_LE = np.linalg.eigvalsh(LE)

    df["LE_principal_min"] = principal_LE[:, 0]
    df["LE_principal_mid"] = principal_LE[:, 1]
    df["LE_principal_max"] = principal_LE[:, 2]

    # Principal stretches
    df["lambda_min"] = np.exp(df["LE_principal_min"])
    df["lambda_mid"] = np.exp(df["LE_principal_mid"])
    df["lambda_max"] = np.exp(df["LE_principal_max"])

    # Principal nominal strains
    df["nominal_principal_min"] = df["lambda_min"] - 1.0
    df["nominal_principal_mid"] = df["lambda_mid"] - 1.0
    df["nominal_principal_max"] = df["lambda_max"] - 1.0

    # Uniaxial-style tensile/compressive extrema
    df["nominal_tension_max"] = np.maximum(df["nominal_principal_max"], 0.0)

    df["nominal_compression"] = np.minimum(df["nominal_principal_min"], 0.0)
    df["nominal_compression_magnitude"] = -df["nominal_compression"]

    # Biaxial indicators
    # If the middle principal strain is positive, at least two principal strains are tensile.
    df["nominal_biaxial_tension"] = np.maximum(df["nominal_principal_mid"], 0.0)

    # If the middle principal strain is negative, at least two principal strains are compressive.
    df["nominal_biaxial_compression"] = np.minimum(df["nominal_principal_mid"], 0.0)
    df["nominal_biaxial_compression_magnitude"] = -df["nominal_biaxial_compression"]

    # Shear-style measures
    df["nominal_max_shear"] = (
        df["nominal_principal_max"] - df["nominal_principal_min"]
    )

    df["nominal_max_tensor_shear"] = 0.5 * df["nominal_max_shear"]

    # Planar shear-type indicator:
    # nonzero only when there is one tensile and one compressive principal strain.
    df["nominal_planar_shear"] = np.where(
        (df["nominal_principal_max"] > 0.0)
        & (df["nominal_principal_min"] < 0.0),
        df["nominal_max_shear"],
        0.0,
    )

    df["nominal_planar_tensor_shear"] = 0.5 * df["nominal_planar_shear"]

    return df

In [85]:
inp_dict = {
    'sub': '14548R',
    'study': '6e',
    'run_id': '00',
    'run_id_mesh': '35T',
    'pose': 'neutral',
    'bone': 'tpm',
    'step': 0,
    'frame': -1,
    'field': 'LE',
}

strain = get_field_df_wrap(**inp_dict)
df = add_strain_measures_from_LE(strain)

summary = {
    "min_principal_LE": df["LE_principal_min"].min(),
    "max_principal_LE": df["LE_principal_max"].max(),

    "min_lambda": df["lambda_min"].min(),
    "max_lambda": df["lambda_max"].max(),

    "max_nominal_tension": df["nominal_tension_max"].max(),
    "max_nominal_compression_magnitude": df["nominal_compression_magnitude"].max(),

    "max_biaxial_tension": df["nominal_biaxial_tension"].max(),
    "max_biaxial_compression_magnitude": df["nominal_biaxial_compression_magnitude"].max(),
    
    "max_planar_shear": df["nominal_planar_shear"].max(),
}
summary

{'min_principal_LE': np.float64(-2.301396840513404),
 'max_principal_LE': np.float64(1.9264350054878856),
 'min_lambda': np.float64(0.10011889587323278),
 'max_lambda': np.float64(6.864992903433153),
 'max_nominal_tension': np.float64(5.864992903433153),
 'max_nominal_compression_magnitude': np.float64(0.8998811041267673),
 'max_biaxial_tension': np.float64(1.1156672350256125),
 'max_biaxial_compression_magnitude': np.float64(0.4329374429828152),
 'max_planar_shear': np.float64(6.667831623162748)}

In [86]:
np.percentile(df["nominal_tension_max"], 90)

np.float64(0.9238570822492168)